In [ ]:
try:
  import google.colab
  IN_COLAB = True
  print("Running as a Colab notebook")
  %pip install git+https://github.com/neelnanda-io/Easy-Transformer.git@clean-transformer-demo
  # Install another version of node that makes PySvelte work way faster
  !curl -fsSL https://deb.nodesource.com/setup_16.x | sudo -E bash -; sudo apt-get install -y nodejs
  %pip install git+https://github.com/neelnanda-io/PySvelte.git
  %pip install fancy_einsum
  %pip install einops
except:
  IN_COLAB = False
  print("Running as a Jupyter notebook - intended for development only!")

In [ ]:
%pip install fancy_einsum
%pip install einops

In [ ]:
%pip install torch torchvision torchaudio


In [1]:
import einops
from fancy_einsum import einsum
from dataclasses import dataclass
#from easy_transformer import EasyTransformer
import torch
import torch.nn as nn
import numpy as np
import math
#from easy_transformer.utils import get_corner, gelu_new, tokenize_and_concatenate
import tqdm.auto as tqdm

/Users/kalle/proj/asiaat/adk-voice-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from easy_transformer import EasyTransformer


In [7]:
#from easy_transformer.utils import get_corner, gelu_new, tokenize_and_concatenate
from transformer_lens.utils import get_corner


In [12]:
from transformer_lens.utils import gelu_new


In [14]:
from transformer_lens.utils import tokenize_and_concatenate

In [16]:
import torch
print(torch.backends.mps.is_available())

True


In [18]:

from transformer_lens import HookedTransformer

In [19]:
# Seade: kas MPS või CPU fallback
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Lae mudel
reference_gpt2 = HookedTransformer.from_pretrained(
    "gpt2-small", 
    fold_ln=False, 
    center_unembed=False, 
    center_writing_weights=False
)

Loaded pretrained model gpt2-small into HookedTransformer


In [20]:
# Liiguta mudel MPS-ile
reference_gpt2.to(device)

Moving model to device:  mps


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (hook_re

In [22]:
sorted_vocab = sorted(list(reference_gpt2.tokenizer.vocab.items()), key=lambda n:n[1])
print(sorted_vocab[:20])
print()
print(sorted_vocab[250:270])
print()
print(sorted_vocab[990:1010])
print()

[('!', 0), ('"', 1), ('#', 2), ('$', 3), ('%', 4), ('&', 5), ("'", 6), ('(', 7), (')', 8), ('*', 9), ('+', 10), (',', 11), ('-', 12), ('.', 13), ('/', 14), ('0', 15), ('1', 16), ('2', 17), ('3', 18), ('4', 19)]

[('ľ', 250), ('Ŀ', 251), ('ŀ', 252), ('Ł', 253), ('ł', 254), ('Ń', 255), ('Ġt', 256), ('Ġa', 257), ('he', 258), ('in', 259), ('re', 260), ('on', 261), ('Ġthe', 262), ('er', 263), ('Ġs', 264), ('at', 265), ('Ġw', 266), ('Ġo', 267), ('en', 268), ('Ġc', 269)]

[('Ġprodu', 990), ('Ġstill', 991), ('led', 992), ('ah', 993), ('Ġhere', 994), ('Ġworld', 995), ('Ġthough', 996), ('Ġnum', 997), ('arch', 998), ('imes', 999), ('ale', 1000), ('ĠSe', 1001), ('ĠIf', 1002), ('//', 1003), ('ĠLe', 1004), ('Ġret', 1005), ('Ġref', 1006), ('Ġtrans', 1007), ('ner', 1008), ('ution', 1009)]



In [23]:
sorted_vocab[-20:]

[('Revolution', 50237),
 ('Ġsnipers', 50238),
 ('Ġreverted', 50239),
 ('Ġconglomerate', 50240),
 ('Terry', 50241),
 ('794', 50242),
 ('Ġharsher', 50243),
 ('Ġdesolate', 50244),
 ('ĠHitman', 50245),
 ('Commission', 50246),
 ('Ġ(/', 50247),
 ('âĢ¦."', 50248),
 ('Compar', 50249),
 ('Ġamplification', 50250),
 ('ominated', 50251),
 ('Ġregress', 50252),
 ('ĠCollider', 50253),
 ('Ġinformants', 50254),
 ('Ġgazed', 50255),
 ('<|endoftext|>', 50256)]

In [24]:
print(reference_gpt2.to_tokens("Whether a word begins with a capital or space matters!"))
print(reference_gpt2.to_tokens("Whether a word begins with a capital or space matters!", prepend_bos=False))

tensor([[50256, 15354,   257,  1573,  6140,   351,   257,  3139,   393,  2272,
          6067,     0]], device='mps:0')
tensor([[15354,   257,  1573,  6140,   351,   257,  3139,   393,  2272,  6067,
             0]], device='mps:0')


In [25]:
print(reference_gpt2.to_str_tokens("Ralph"))
print(reference_gpt2.to_str_tokens(" Ralph"))
print(reference_gpt2.to_str_tokens(" ralph"))
print(reference_gpt2.to_str_tokens("ralph"))

['<|endoftext|>', 'R', 'alph']
['<|endoftext|>', ' Ralph']
['<|endoftext|>', ' r', 'alph']
['<|endoftext|>', 'ral', 'ph']


In [26]:
reference_gpt2.to_str_tokens("56873+3184623=123456789-1000000000")

['<|endoftext|>',
 '568',
 '73',
 '+',
 '318',
 '46',
 '23',
 '=',
 '123',
 '45',
 '67',
 '89',
 '-',
 '1',
 '000000',
 '000']

In [27]:
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = reference_gpt2.to_tokens(reference_text)
print(tokens)
print(tokens.shape)
print(reference_gpt2.to_str_tokens(tokens))

tensor([[50256,    40,   716,   281,  4998,  1960,   382, 19741,    11,   875,
         12342,    12,  8807,    11,   402, 11571,    12,    17,  3918, 47385,
            13,  1881,  1110,   314,   481,  7074,  1692,  1241,  4430,   290,
          1011,   625,   262,   995,     0]], device='mps:0')
torch.Size([1, 35])
['<|endoftext|>', 'I', ' am', ' an', ' amazing', ' aut', 'ore', 'gressive', ',', ' dec', 'oder', '-', 'only', ',', ' G', 'PT', '-', '2', ' style', ' transformer', '.', ' One', ' day', ' I', ' will', ' exceed', ' human', ' level', ' intelligence', ' and', ' take', ' over', ' the', ' world', '!']


In [29]:
# Liiguta tokenid õigesse seadmesse
tokens = tokens.to(device)

# Käivita mudel tokenitega
logits, cache = reference_gpt2.run_with_cache(tokens)

# Väljasta logits mõõtmed
print(logits.shape)

torch.Size([1, 35, 50257])


In [30]:
log_probs = logits.log_softmax(dim=-1)
probs = logits.log_softmax(dim=-1)
print(log_probs.shape)
print(probs.shape)

torch.Size([1, 35, 50257])
torch.Size([1, 35, 50257])


In [31]:
list(zip(reference_gpt2.to_str_tokens(reference_text), reference_gpt2.tokenizer.batch_decode(logits.argmax(dim=-1)[0])))

[('<|endoftext|>', '\n'),
 ('I', "'m"),
 (' am', ' a'),
 (' an', ' avid'),
 (' amazing', ' person'),
 (' aut', 'od'),
 ('ore', 'sp'),
 ('gressive', '.'),
 (',', ' and'),
 (' dec', 'ently'),
 ('oder', ','),
 ('-', 'driven'),
 ('only', ' programmer'),
 (',', ' and'),
 (' G', 'IM'),
 ('PT', '-'),
 ('-', 'only'),
 ('2', '.'),
 (' style', ','),
 (' transformer', '.'),
 ('.', ' I'),
 (' One', ' of'),
 (' day', ' I'),
 (' I', ' will'),
 (' will', ' be'),
 (' exceed', ' my'),
 (' human', 'ly'),
 (' level', ' of'),
 (' intelligence', ' and'),
 (' and', ' I'),
 (' take', ' over'),
 (' over', ' the'),
 (' the', ' world'),
 (' world', '.'),
 ('!', ' I')]

In [37]:
next_token = logits[0, -1].argmax(dim=-1)
print(next_token)

# Teisenda stringiks
next_token_str = reference_gpt2.tokenizer.decode(next_token)
print("Next token as string:", next_token_str)

tensor(314, device='mps:0')
Next token as string:  I


In [34]:
next_tokens = torch.cat([tokens, torch.tensor(next_token, device=device, dtype=torch.int64)[None, None]], dim=-1)
new_logits = reference_gpt2(next_tokens)
print("New Input:", next_tokens)
print(next_tokens.shape)
print("New Input:", reference_gpt2.tokenizer.decode(next_tokens[0]))

print(new_logits.shape)
print(new_logits[-1, -1].argmax(-1))

print(reference_gpt2.tokenizer.decode(new_logits[-1, -1].argmax(-1)))


/var/folders/9l/05gqvbh53r758m_g76k5fhnm0000gn/T/ipykernel_31722/669351975.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  next_tokens = torch.cat([tokens, torch.tensor(next_token, device=device, dtype=torch.int64)[None, None]], dim=-1)


New Input: tensor([[50256,    40,   716,   281,  4998,  1960,   382, 19741,    11,   875,
         12342,    12,  8807,    11,   402, 11571,    12,    17,  3918, 47385,
            13,  1881,  1110,   314,   481,  7074,  1692,  1241,  4430,   290,
          1011,   625,   262,   995,     0,   314]], device='mps:0')
torch.Size([1, 36])
New Input: <|endoftext|>I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world! I
torch.Size([1, 36, 50257])
tensor(716, device='mps:0')
 am


Key:
```
batch = 1
position = 35
d_model = 768
n_heads = 12
n_layers = 12
d_mlp = 3072 (4 * d_model)
d_head = 64 (d_model / n_heads)
```

In [44]:
for activation_name, activation in cache.cache_dict.items():
    # Only print for first layer
    if ".0." in activation_name or "blocks" not in activation_name:
        print(activation_name, activation.shape)

hook_embed torch.Size([1, 35, 768])
hook_pos_embed torch.Size([1, 35, 768])
blocks.0.hook_resid_pre torch.Size([1, 35, 768])
blocks.0.ln1.hook_scale torch.Size([1, 35, 1])
blocks.0.ln1.hook_normalized torch.Size([1, 35, 768])
blocks.0.attn.hook_q torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_k torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_v torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_attn_scores torch.Size([1, 12, 35, 35])
blocks.0.attn.hook_pattern torch.Size([1, 12, 35, 35])
blocks.0.attn.hook_z torch.Size([1, 35, 12, 64])
blocks.0.hook_attn_out torch.Size([1, 35, 768])
blocks.0.hook_resid_mid torch.Size([1, 35, 768])
blocks.0.ln2.hook_scale torch.Size([1, 35, 1])
blocks.0.ln2.hook_normalized torch.Size([1, 35, 768])
blocks.0.mlp.hook_pre torch.Size([1, 35, 3072])
blocks.0.mlp.hook_post torch.Size([1, 35, 3072])
blocks.0.hook_mlp_out torch.Size([1, 35, 768])
blocks.0.hook_resid_post torch.Size([1, 35, 768])
ln_final.hook_scale torch.Size([1, 35, 1])
ln_final.hook_normalized torc

In [71]:

from dataclasses import dataclass
import torch
import torch.nn as nn
import einops

# ----- Konfiguratsioon -----
@dataclass
class Config:
    d_model: int = 768
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12
    debug: bool = True  # Võid vajadusel panna True

cfg = Config()
print(cfg)

Config(d_model=768, layer_norm_eps=1e-05, d_vocab=50257, init_range=0.02, n_ctx=1024, d_head=64, d_mlp=3072, n_heads=12, n_layers=12, debug=True)


In [75]:
class LayerNorm(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.w = nn.Parameter(torch.ones(cfg.d_model))
        self.b = nn.Parameter(torch.zeros(cfg.d_model))

    def forward(self, residual):
        # residual: [batch, position, d_model]
        if self.cfg.debug:
            print("Residual:", residual.shape)

        # Lahuta keskmine per token
        mean = einops.reduce(residual, "b p d -> b p 1", "mean")
        residual = residual - mean

        # Variaans ja skaleerimine
        var = einops.reduce(residual.pow(2), "b p d -> b p 1", "mean")
        scale = (var + self.cfg.layer_norm_eps).sqrt()

        # Normeeritud väljund
        normalized = residual / scale
        normalized = normalized * self.w + self.b

        if self.cfg.debug:
            print("Normalized:", normalized.shape)

        return normalized

Tests are great, write lightweight ones to use as you go!

**Naive test:** Generate random inputs of the right shape, input to your model, check whether there's an error and print the correct output.

In [48]:
def rand_float_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    random_input = torch.randn(shape).cuda()
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()
    return output

def rand_int_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    random_input = torch.randint(100, 1000, shape).cuda()
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()
    return output

def load_gpt2_test(cls, gpt2_layer, input_name, cache_dict=cache.cache_dict):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    layer.load_state_dict(gpt2_layer.state_dict(), strict=False)
    # Allow inputs of strings or tensors
    if isinstance(input_name, str): 
        reference_input = cache_dict[input_name]
    else:
        reference_input = input_name
    print("Input shape:", reference_input.shape)
    output = layer(reference_input)
    print("Output shape:", output.shape)
    reference_output = gpt2_layer(reference_input)
    print("Reference output shape:", reference_output.shape)

    comparison = torch.isclose(output, reference_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum()/comparison.numel():.2%} of the values are correct")
    return output

In [76]:
def rand_float_mac_test(cls, shape, conf):
    # Automaatne seadme valik: MPS -> CPU
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    layer = cls(conf).to(device)
    random_input = torch.randn(shape).to(device)

    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()
    return output

In [77]:
rand_float_mac_test(LayerNorm, [2, 4, 768],cfg)

Input shape: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])



tensor([[[-0.8589,  0.9320, -1.7730,  ..., -0.8076,  0.6817,  0.4093],
         [ 0.8351, -1.7028,  0.2007,  ...,  0.3039,  0.8438, -0.3990],
         [-0.3864,  1.2010, -0.8570,  ...,  1.9928, -0.8163, -0.0760],
         [-0.0793, -0.1000,  1.5855,  ..., -1.4209,  0.1994, -0.0521]],

        [[-1.0029,  0.5053, -0.3894,  ...,  0.7162, -0.0186,  0.0766],
         [-1.1926,  1.0706,  0.5136,  ..., -0.2006, -0.9139,  0.5955],
         [-0.0460,  0.0134, -0.0410,  ...,  0.9206, -0.6110, -2.7458],
         [ 0.4868,  0.7244,  0.8960,  ..., -1.6086,  0.3452, -0.2400]]],
       device='mps:0', grad_fn=<AddBackward0>)

In [104]:
def rand_int_mac_test(cls, shape, conf):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    layer = cls(conf).to(device)
    random_input = torch.randint(100, 1000, shape).float().to(device)  # <- float()

    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()
    return output



In [105]:
rand_int_mac_test(LayerNorm, [2, 4, 768], cfg)


Input shape: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])



tensor([[[-0.6235, -1.0150,  1.4538,  ..., -0.3288, -0.0678, -1.3171],
         [-0.9268, -0.8509,  0.4357,  ..., -0.2816, -0.7560, -1.3936],
         [-0.3144, -0.3223, -1.4460,  ...,  0.8489, -0.3381,  1.4068],
         [ 0.0228,  0.0843,  0.0036,  ..., -1.3477, -1.0943, -0.8909]],

        [[ 0.4552, -0.4221, -1.6240,  ..., -0.1090, -0.3100,  0.7296],
         [ 1.5783, -1.0821, -0.1461,  ...,  0.0131,  1.2676, -0.7675],
         [ 1.7114, -0.7591, -0.7517,  ...,  0.5285,  1.6964,  0.4799],
         [-1.3154,  1.2831,  1.5197,  ..., -0.2644,  0.2515, -0.5785]]],
       device='mps:0', grad_fn=<AddBackward0>)

In [93]:

def load_gpt2_mac_test(cfg, cls, gpt2_layer, input_name, cache_dict):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    # Loo enda layer ja GPT-2 layer õigele seadmele
    layer = cls(cfg).to(device)
    gpt2_layer = gpt2_layer.to(device)  # ← see oli puudu

    # Lae parameetrid (gpt2_layer juba seadmel)
    layer.load_state_dict(gpt2_layer.state_dict(), strict=False)

    # Võta sisend
    if isinstance(input_name, str):
        reference_input = cache_dict[input_name]
    else:
        reference_input = input_name

    reference_input = reference_input.to(device)

    print("Input shape:", reference_input.shape)

    # Arvuta mõlemad väljundid
    output = layer(reference_input)
    print("Output shape:", output.shape)

    reference_output = gpt2_layer(reference_input)
    print("Reference output shape:", reference_output.shape)

    # Võrdlus
    comparison = torch.isclose(output, reference_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum()/comparison.numel():.2%} of the values are correct")

    return output



In [94]:
# Näiteks GPT-2 mudelist võetud nn.LayerNorm
gpt2_layer = nn.LayerNorm(768)

# Simuleeritud sisend (näiteks "blocks.11.hook_resid_post")
reference_input = torch.randn(2, 4, 768)

# Simuleeritud cache
cache = {
    "blocks.11.hook_resid_post": reference_input
}

In [95]:
_ = load_gpt2_mac_test(cfg, LayerNorm, gpt2_layer, "blocks.11.hook_resid_post", cache)

Input shape: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])
Reference output shape: torch.Size([2, 4, 768])
100.00% of the values are correct


# Embeddings

In [106]:
class Embed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_E = nn.Parameter(torch.empty((cfg.d_vocab, cfg.d_model)))
        nn.init.normal_(self.W_E, std=self.cfg.init_range)
    
    def forward(self, tokens):
        # tokens: [batch, position]
        if cfg.debug: print("Tokens:", tokens.shape)
        embed = self.W_E[tokens, :] # [batch, position, d_model]
        if cfg.debug: print("Embeddings:", embed.shape)
        return embed




In [108]:
def rand_int_mac_test_2(cls, shape, cfg):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    layer = cls(cfg).to(device)

    # GENEREERI TÄISARVULINE SISEND (EI float()!)
    random_input = torch.randint(100, 1000, shape).to(device)  # ← PARANDATUD

    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()


In [109]:
rand_int_mac_test_2(Embed, [2, 4],cfg)

Input shape: torch.Size([2, 4])
Tokens: torch.Size([2, 4])
Embeddings: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])



In [110]:
class PosEmbed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_pos = nn.Parameter(torch.empty((cfg.n_ctx, cfg.d_model)))
        nn.init.normal_(self.W_pos, std=self.cfg.init_range)

    def forward(self, tokens):
        # tokens: [batch, position]
        if cfg.debug: print("Tokens:", tokens.shape)
        pos_embed = self.W_pos[:tokens.size(1), :] # [position, d_model]
        pos_embed = einops.repeat(pos_embed, "position d_model -> batch position d_model", batch=tokens.size(0))
        if cfg.debug: print("pos_embed:", pos_embed.shape)
        return pos_embed

In [111]:
rand_int_mac_test_2(PosEmbed, [2, 4],cfg)

Input shape: torch.Size([2, 4])
Tokens: torch.Size([2, 4])
pos_embed: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])



## Attention is all you need

* **Step 1:** Produce an attention pattern - for each destination token, probability distribution over previous tokens (incl current token)
    * Linear map from input -> query, key shape [batch, position, head_index, d_head]
    * Dot product every *pair* of queries and keys to get attn_scores [batch, head_index, query_pos, key_pos] (query = dest, key = source)
    * Scale and mask attn_scores to make it lower triangular, ie causal
    * softmax row-wise, to get a probability distribution along each the key_pos dimension - this is our attention pattern!
* **Step 2:** Move information from source tokens to destination token using attention pattern (move = apply linear map)
    * Linear map from input -> value [batch, key_pos, head_index, d_head]
    * Mix along the key_pos with attn pattern to get z, a mixed value [batch, query_pos, head_index, d_head]
    * Map to output, [batch, position, d_model] (position = query_pos, we've summed over all heads)

In [120]:
%pip install unseal


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
96161.79s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Using cached Unseal-0.2.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached matplotlib-3.10.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached streamlit-1.47.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached contourpy-1.3.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.59.0-cp311-cp311-macosx_10_9_universal2.whl.metadata (107 kB)
  Using cached kiwisolver-1.4.8-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.2 kB)
  Using cached altair-5.5.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached jsonschema-4.25.0-py3-none-any.whl.metadata (7.7 kB)
  Using cached narwhals-1.47.1-py3-none-any.whl.metadata (11 kB)
  Using c

In [ ]:
import pysvelte
pysvelte.AttentionMulti(tokens=reference_gpt2.to_str_tokens(reference_text), attention=cache['blocks.0.attn.hook_attn'][0].permute(1, 2, 0)).show()

In [131]:
class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        nn.init.normal_(self.W_Q, std=self.cfg.init_range)
        self.b_Q = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
        self.W_K = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        nn.init.normal_(self.W_K, std=self.cfg.init_range)
        self.b_K = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
        self.W_V = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        nn.init.normal_(self.W_V, std=self.cfg.init_range)
        self.b_V = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
        
        self.W_O = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
        nn.init.normal_(self.W_O, std=self.cfg.init_range)
        self.b_O = nn.Parameter(torch.zeros((cfg.d_model)))
        
        self.register_buffer("IGNORE", torch.tensor(-1e5, dtype=torch.float32, device="mps"))
    
    def forward(self, normalized_resid_pre):
        # normalized_resid_pre: [batch, position, d_model]
        if cfg.debug: print("Normalized_resid_pre:", normalized_resid_pre.shape)

        q = einsum("batch query_pos d_model, n_heads d_model d_head -> batch query_pos n_heads d_head", normalized_resid_pre, self.W_Q) + self.b_Q
        k = einsum("batch key_pos d_model, n_heads d_model d_head -> batch key_pos n_heads d_head", normalized_resid_pre, self.W_K) + self.b_K

        attn_scores = einsum("batch query_pos n_heads d_head, batch key_pos n_heads d_head -> batch n_heads query_pos key_pos", q, k)
        attn_scores = attn_scores / math.sqrt(self.cfg.d_head)
        attn_scores = self.apply_causal_mask(attn_scores)

        attn = attn_scores.softmax(dim=-1) # [batch, n_head, query_pos, key_pos]

        v = einsum("batch key_pos d_model, n_heads d_model d_head -> batch key_pos n_heads d_head", normalized_resid_pre, self.W_V) + self.b_V

        z = einsum("batch n_heads query_pos key_pos, batch key_pos n_heads d_head -> batch query_pos n_heads d_head", attn, v)

        attn_out = einsum("batch query_pos n_heads d_head, n_heads d_head d_model -> batch query_pos d_model", z, self.W_O) + self.b_O
        return attn_out 
        
    def apply_causal_mask(self, attn_scores):
        # attn_scores: [batch, n_head...]
        mask = torch.triu(torch.ones(attn_scores.size(-2), attn_scores.size(-1),
                        device=attn_scores.device), diagonal=1).bool()
        attn_scores.masked_fill_(mask, self.IGNORE)
        return attn_scores

        


In [142]:
def rand_int_mac_test_3(cls, shape, cfg):
    import torch

    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    layer = cls(cfg).to(device)

    full_shape = shape + [cfg.d_model]  # näiteks [2, 4, 768]
    random_input = torch.randn(full_shape).to(device)

    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)

    # Prindi väike osa väljundist (nt 3 tokenit, 3 mõõdet igaühes)
    print(output[:, :3, :3])  # [batch, 3, 3]
    
  



In [143]:
rand_int_mac_test_3(Attention, [2, 4], cfg)


Input shape: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])
tensor([[[ 0.4527,  0.4612, -0.0316],
         [ 0.3472,  0.2236, -0.0334],
         [ 0.2432,  0.3188,  0.0899]],

        [[-0.0685,  0.6483, -0.2014],
         [ 0.0747,  0.1977, -0.3812],
         [-0.0990,  0.1416, -0.2658]]], device='mps:0',
       grad_fn=<SliceBackward0>)
